In [1]:
pip install pandas numpy ydata-profiling pandera scikit-learn dvc mlflow kaggle

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/924.3 kB ? eta -:--:--
   ---------------------------------------- 924.3/924.3 kB 14.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   --------------------------------- ------ 1.0/1.3 MB 7.2 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 6.5 MB/s  0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144614 sha256=0d290a993ad1d74dcd91300ca2bdee1dfb1fc1db71146fa2d1af6f85fe04ae43
  Stored in directory: c:\users\vijayendra\appdata\local\pip\cache\wheels\23\cf\80\f3efa822e6ab23277902ee9165fe772eeb1df

In [3]:
# Cell 2: Module Imports
import os
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import pandera as pa

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from ydata_profiling import ProfileReport

# Create required project directory structure inside Jupyter
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports", exist_ok=True)

In [4]:
# Cell 3: Data Ingestion
# Assuming sales_data.csv is uploaded to your working directory or data/raw/
df_raw = pd.read_csv("sales_data.csv")

# Save unmodified raw copy
df_raw.to_csv("data/raw/sales_data.csv", index=False)

# Required outputs
print("First 5 records:\n", df_raw.head())
print(f"\nDataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print("\nColumn Names:", df_raw.columns.tolist())
print("\nData Types:\n", df_raw.dtypes)

First 5 records:
    Product_ID   Sale_Date Sales_Rep Region  Sales_Amount  Quantity_Sold  \
0        1052  2023-02-03       Bob  North       5053.97             18   
1        1093  2023-04-21       Bob   West       4384.02             17   
2        1015  2023-09-21     David  South       4631.23             30   
3        1072  2023-08-24       Bob  South       2167.94             39   
4        1061  2023-03-24   Charlie   East       3750.20             13   

  Product_Category  Unit_Cost  Unit_Price Customer_Type  Discount  \
0        Furniture     152.75      267.22     Returning      0.09   
1        Furniture    3816.39     4209.44     Returning      0.11   
2             Food     261.56      371.40     Returning      0.20   
3         Clothing    4330.03     4467.75           New      0.02   
4      Electronics     637.37      692.71           New      0.08   

  Payment_Method Sales_Channel Region_and_Sales_Rep  
0           Cash        Online            North-Bob  
1       

In [5]:
# Cell 4: YData Profiling
profile = ProfileReport(
    df_raw, title="Sales Dataset Profiling Report", explorative=True
)
profile.to_file("reports/sales_profiling_report.html")

# Display inline within Jupyter Lab
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 14/14 [00:00<00:00, 159.66it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
# Explicitly cast to float before validating
df_raw["Quantity_Sold"] = df_raw["Quantity_Sold"].astype(float)

# Now validate
validated_df = sales_schema.validate(df_raw, lazy=True)

In [7]:
# Cell 5: Data Validation Schema
sales_schema = pa.DataFrameSchema({
    "Quantity_Sold": pa.Column(float, pa.Check.ge(0), nullable=True),
    "Unit_Cost": pa.Column(float, pa.Check.ge(0), nullable=True),
    "Unit_Price": pa.Column(float, pa.Check.ge(0), nullable=True),
    "Discount": pa.Column(
        float, pa.Check.in_range(0, 1), nullable=True
    ),  # expected range 0-100%
    "Sales_Amount": pa.Column(float, pa.Check.ge(0), nullable=True),
})

# Perform lazy validation
try:
    validated_df = sales_schema.validate(df_raw, lazy=True)
    print("Data Validation Passed!")
except pa.errors.SchemaErrors as err:
    print("Validation errors detected:\n", err.failure_cases)

Data Validation Passed!


C:\Users\Vijayendra\AppData\Local\Programs\Python\Python39\lib\site-packages\pandera\_pandas_deprecated.py:149: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [17]:
# Cell 6: Pipeline Modular Functions
def ingest_data(filepath):
    return pd.read_csv(filepath)


def validate_data_stage(df):
    return sales_schema.validate(df, lazy=True)


def clean_data(df):
    df_clean = df.drop_duplicates().copy()
    if "Date" in df_clean.columns:
        df_clean["Date"] = pd.to_datetime(df_clean["Date"])
    return df_clean


def transform_data(df):
    df_transformed = df.copy()

    # Time Features
    if "Date" in df_transformed.columns:
        df_transformed["Year"] = df_transformed["Date"].dt.year
        df_transformed["Month"] = df_transformed["Date"].dt.month
        df_transformed["DayOfWeek"] = df_transformed["Date"].dt.dayofweek

    # Engineered Financial Signals (Without target leakage)
    if (
        "Unit_Price" in df_transformed.columns
        and "Quantity_Sold" in df_transformed.columns
    ):
        df_transformed["Gross_Sales"] = (
            df_transformed["Unit_Price"] * df_transformed["Quantity_Sold"]
        )
    if (
        "Unit_Cost" in df_transformed.columns
        and "Quantity_Sold" in df_transformed.columns
    ):
        df_transformed["Total_Cost"] = (
            df_transformed["Unit_Cost"] * df_transformed["Quantity_Sold"]
        )

    return df_transformed


def save_processed_data(df, path):
    df.to_csv(path, index=False)


# Main Pipeline Execution
def run_pipeline():
    raw_data = ingest_data("data/raw/sales_data.csv")
    cleaned_data = clean_data(raw_data)
    transformed_data = transform_data(cleaned_data)
    save_processed_data(transformed_data, "data/processed/sales_processed.csv")
    print("Full Pipeline executed successfully!")
    return transformed_data


df_processed = run_pipeline()

Full Pipeline executed successfully!


In [18]:
# Cell 7: DVC Track Commands
!git init
!dvc init
!dvc add data/raw/sales_data.csv
!dvc add data/processed/sales_processed.csv
!git add .gitignore data/raw/sales_data.csv.dvc data/processed/sales_processed.csv.dvc
!git commit -m "Track dataset versions using DVC"
!dvc status

Reinitialized existing Git repository in C:/Users/Vijayendra/eclipse-workspace/vector 3D/.git/


ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.



To track the changes with git, run:

	git add 'data\raw\sales_data.csv.dvc'

To enable auto staging, run:

	dvc config core.autostage true


\u280b Checking graph




To track the changes with git, run:

	git add 'data\processed\sales_processed.csv.dvc'

To enable auto staging, run:

	dvc config core.autostage true


\u280b Checking graph

fatal: pathspec '.gitignore' did not match any files


On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Untitled3.ipynb
	modified:   reports/sales_profiling_report.html

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	mlruns/942814182249313971/6288da21c4594ae8b18eca8862d57dfb/
	mlruns/942814182249313971/models/m-6e68b5ccc66849b38e48900e5b359fcc/

no changes added to commit (use "git add" and/or "git commit -a")
Data and pipelines are up to date.


In [20]:
# Cell 10: Commit the generated .dvc pointer files to Git
!git add data/raw/sales_data.csv.dvc data/raw/.gitignore
!git add data/processed/sales_processed.csv.dvc data/processed/.gitignore
!git commit -m "Add DVC pointer files for raw and processed datasets"

# Verify DVC tracking status
!dvc status

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Untitled3.ipynb
	modified:   reports/sales_profiling_report.html

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	mlruns/942814182249313971/6288da21c4594ae8b18eca8862d57dfb/
	mlruns/942814182249313971/models/m-6e68b5ccc66849b38e48900e5b359fcc/

no changes added to commit (use "git add" and/or "git commit -a")
Data and pipelines are up to date.


In [22]:
# Drop non-predictive ID columns and raw text dates to prevent high-cardinality noise
cols_to_drop = ["Sales_Amount"]

# Add any ID or raw date string columns if present in your dataset
for col in ["ID", "Transaction_ID", "Date", "Customer_ID"]:
    if col in df.columns:
        cols_to_drop.append(col)

X = df.drop(columns=cols_to_drop)
y = df["Sales_Amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
# Cell 8: Model Training & Experiment Tracking
# Load processed dataset
df = pd.read_csv("data/processed/sales_processed.csv")

# Stage 8: Target Separation (Avoid target leakage)
X = df.drop(columns=["Sales_Amount"])
y = df["Sales_Amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Detect feature types
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Stage 9: Preprocessing Pipeline Construction
num_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols),
    ]
)

# Stage 10 & 11: Model Building & MLflow Tracking
mlflow.set_experiment("Sales_Amount_Prediction")

with mlflow.start_run():
    n_estimators = 100
    random_state = 42

    # Combined Preprocessing + Estimation Pipeline
    model_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "regressor",
                RandomForestRegressor(
                    n_estimators=n_estimators, random_state=random_state
                ),
            ),
        ]
    )

    # Train
    model_pipeline.fit(X_train, y_train)

    # Predict & Evaluate
    y_pred = model_pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Log to MLflow
    mlflow.log_param("model_name", "RandomForestRegressor")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("random_state", random_state)

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2_Score", r2)

    mlflow.sklearn.log_model(model_pipeline, "model")

    print(f"Results -> MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2 Score: {r2:.4f}")

2026/09/01 13:20:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 13:20:22 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Results -> MAE: 2816.82 | RMSE: 3218.86 | R2 Score: -0.1212


In [25]:
# Import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

with mlflow.start_run():
    max_depth = 10
    random_state = 42

    # Combined Preprocessing + DecisionTree Pipeline
    dt_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "regressor",
                DecisionTreeRegressor(
                    max_depth=max_depth, random_state=random_state
                ),
            ),
        ]
    )

    # Train Decision Tree Model
    dt_pipeline.fit(X_train, y_train)

    # Predict & Evaluate
    y_pred_dt = dt_pipeline.predict(X_test)
    mae_dt = mean_absolute_error(y_test, y_pred_dt)
    mse_dt = mean_squared_error(y_test, y_pred_dt)
    rmse_dt = np.sqrt(mse_dt)
    r2_dt = r2_score(y_test, y_pred_dt)

    # Log Model Name and Hyperparameters
    mlflow.log_param("model_name", "DecisionTreeRegressor")
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)

    # Log Metrics (including MSE)
    mlflow.log_metric("MSE", mse_dt)
    mlflow.log_metric("MAE", mae_dt)
    mlflow.log_metric("RMSE", rmse_dt)
    mlflow.log_metric("R2_Score", r2_dt)

    # Log Model Artifact
    mlflow.sklearn.log_model(dt_pipeline, "model")

    print(
        f"Decision Tree -> MSE: {mse_dt:.2f} | MAE: {mae_dt:.2f} | R2: {r2_dt:.4f}"
    )

2026/09/01 13:27:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 13:27:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Decision Tree -> MSE: 10411608.93 | MAE: 2821.37 | R2: -0.1266


In [ ]:
!mlflow ui --port 5000

In [2]:
# 1. Reset your local branch back to match GitHub (keeps your modified files intact)
!git reset --soft origin/main

# 2. Stage the cleaned notebook and files
!git add .

# 3. Create a fresh, clean commit
!git commit -m "Add MLOps notebook with secrets removed"

# 4. Push to GitHub
!git push origin main

[main 6e54c5f] Add MLOps notebook with secrets removed
 105 files changed, 105610 insertions(+)
 create mode 100644 .classpath
 create mode 100644 .ipynb_checkpoints/Untitled-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/Untitled1-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/Untitled2-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/Untitled3-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/sales_data-checkpoint.csv
 create mode 100644 .ipynb_checkpoints/untitled-checkpoint.txt
 create mode 100644 .project
 create mode 100644 .settings/org.eclipse.core.resources.prefs
 create mode 100644 .settings/org.eclipse.jdt.core.prefs
 create mode 100644 Salary Data.xlsx
 create mode 100644 Untitled.ipynb
 create mode 100644 Untitled1.ipynb
 create mode 100644 Untitled2.ipynb
 create mode 100644 Untitled3.ipynb
 create mode 100644 bin/module-info.class
 create mode 100644 mlruns/0/meta.yaml
 create mode 100644 mlruns/107643745794551697/meta.yaml
 create mode 100

To https://github.com/vijayendra-2006-crypto/mlops-sales-prediction.git
   f33cfb2..6e54c5f  main -> main


In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Set Experiment Context
mlflow.set_experiment("Sales_Amount_Prediction")

# Dictionary to store performance metrics for comparison
model_results = {}

# Define algorithms to compare
models = {
    "DecisionTreeRegressor": DecisionTreeRegressor(max_depth=10, random_state=42),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
}

# 2. Train, Evaluate, and Log Each Model
for model_name, model_instance in models.items():
    with mlflow.start_run(run_name=model_name):
        # Create pipeline with existing preprocessor
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', model_instance)
        ])
        
        # Fit model
        pipeline.fit(X_train, y_train)
        
        # Predict & Evaluate
        y_pred = pipeline.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        
        # Store for tabular comparison
        model_results[model_name] = {
            "MSE": round(mse, 4),
            "RMSE": round(rmse, 4),
            "MAE": round(mae, 4),
            "R2_Score": round(r2, 4)
        }
        
        # Log Hyperparameters & Metrics to MLflow
        mlflow.log_param("model_name", model_name)
        mlflow.log_metric("MSE", mse)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("R2_Score", r2)
        mlflow.sklearn.log_model(pipeline, "model")

# 3. Print Tabular Comparison & Identify Best Model
df_comparison = pd.DataFrame(model_results).T
print("=== Algorithm Performance Comparison ===")
print(df_comparison)

best_model_name = df_comparison['MSE'].idxmin()
print(f"\nBest Model based on lowest MSE: {best_model_name} (MSE = {df_comparison.loc[best_model_name, 'MSE']})")